# GroundX + NVIDIA Quickstart

Ask questions of complex documents and get answers cited to the exact page.

**Before you start:** copy `.env.example` to `.env` and add your `GROUNDX_API_KEY` ([dashboard.groundx.ai](https://dashboard.groundx.ai)) and `NVIDIA_API_KEY` ([build.nvidia.com](https://build.nvidia.com)).

**Self-hosted GroundX instead?** Set `GROUNDX_BASE_URL` in `.env` to your instance (see [`deploy/`](../deploy/)) — every cell below works unchanged.

**Where your data goes:** documents you load here go to your GroundX account (deleted anytime with the last cell). The NVIDIA-hosted model only ever receives retrieved text passages, never your files.

In [ ]:
# 1. Connect
%pip -q install groundx python-dotenv
import os, time
from dotenv import load_dotenv; load_dotenv()
from groundx import GroundX

gx = GroundX(api_key=os.environ["GROUNDX_API_KEY"],
             base_url=os.environ.get("GROUNDX_BASE_URL") or None)  # unset = GroundX cloud
BUCKET_NAME = os.environ.get("GROUNDX_BUCKET", "nvidia-quickstart-demo")
bucket = next((b for b in gx.buckets.list().buckets if b.name == BUCKET_NAME), None)
if bucket is None:
    bucket = gx.buckets.create(name=BUCKET_NAME).bucket
print("bucket:", bucket.bucket_id, BUCKET_NAME)

## 2. Load a document

The sample is the IRS Form 1040 instructions — 100+ pages of dense tables. Swap in any PDF URL. Processing a large document takes a few minutes; the cell polls until it finishes.

In [ ]:
DOC_URL = "https://www.irs.gov/pub/irs-pdf/i1040gi.pdf"   # or your own document URL
name = DOC_URL.rsplit("/", 1)[-1]

already = any(d.file_name == name for d in gx.documents.lookup(id=bucket.bucket_id).documents)
if already:
    print(name, "is already loaded")
else:
    ing = gx.ingest(documents=[{"bucket_id": bucket.bucket_id, "file_name": name,
                                "file_type": "pdf", "source_url": DOC_URL, "process_level": "full"}])
    while True:
        status = gx.documents.get_processing_status_by_id(process_id=ing.ingest.process_id).ingest.status
        print("status:", status)
        if status in ("complete", "error", "cancelled"): break
        time.sleep(20)

## 3. Ask a question, get a cited answer

Every result carries the source file, the page number, and the rectangle on the page it came from.

In [ ]:
r = gx.search.content(id=bucket.bucket_id, query="What is the standard deduction for married filing jointly?", n=3)
top = r.search.results[0]
box = top.bounding_boxes[0] if top.bounding_boxes else None
print("file:", top.file_name)
print("page:", box.page_number if box else "?")
print()
print((top.suggested_text or top.text or "")[:500])

## 4. The same search over plain REST

No SDK required — one POST, with the key in a connection header (never in the request body).

In [ ]:
import requests
resp = requests.post(
    f"https://api.groundx.ai/api/v1/search/{bucket.bucket_id}",
    headers={"X-API-Key": os.environ["GROUNDX_API_KEY"]},
    json={"query": "standard deduction amounts by filing status", "n": 2}, timeout=60)
first = resp.json()["search"]["results"][0]
print(resp.status_code, "| file:", first["fileName"], "| page:", first["boundingBoxes"][0]["pageNumber"])

## 5. Run it as an agent

The same document library, driven by NVIDIA's NeMo Agent Toolkit with a Nemotron model — configured entirely by [`configs/groundx_agent.yml`](../configs/groundx_agent.yml). From a terminal:

```bash
pip install "nvidia-nat[langchain,mcp]"
scripts/run_agent.sh "What is the standard deduction for married filing jointly? Cite the page."
```

The agent finds the bucket, searches it, and answers with the page citation.

## Cleanup

Delete the bucket and every document in it:

In [ ]:
# gx.buckets.delete(bucket_id=bucket.bucket_id)
print("uncomment the line above to delete the bucket and its documents")